In [9]:
from fastai.collab import *
from fastai.tabular.all import *
set_seed(42)

In [10]:
import numpy as np
import pandas as pd

Importing the data

In [11]:
path = untar_data(URLs.ML_100k)

Reading the data

In [12]:
rating_df = pd.read_csv(path/'u.data',delimiter = '\t', header = None, names = ['user','movie','rating','timestamp'])
rating_df.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Reading the dataframe having a movie title

In [13]:
movies_df = pd.read_csv(path/'u.item', delimiter='|',encoding='latin-1',usecols=(0,1), names=('movie','title'), header=None)
movies_df.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


Merging the movies and rating dataframe

In [15]:
rating_df = rating_df.merge(movies_df)  #merging the rating and movie dataframe
rating_df.head()

,user,movie,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


Creation of dataloaders


In [19]:
dls = CollabDataLoaders.from_df(rating_df,item_name = 'title' , bs = 64) #here we are creating a dataloaders based on userID, movie_title as item  and the movie_rating done by the user
dls.show_batch()

,user,title,rating
0,782,Starship Troopers (1997),2
1,943,Judge Dredd (1995),3
2,758,Mission: Impossible (1996),4
3,94,Farewell My Concubine (1993),5
4,23,Psycho (1960),4
5,296,Secrets & Lies (1996),5
6,940,"American President, The (1995)",4
7,334,Star Trek VI: The Undiscovered Country (1991),1
8,380,Braveheart (1995),4
9,690,So I Married an Axe Murderer (1993),1


In [25]:
n_users = len(dls.classes['user'])
n_movies = len(dls.classes['title'])
print('The number of users is:', n_users)
print('The number of movies is:', n_movies)

The number of users is: 944
The number of movies is: 1665


creation of user factor and movie factor

In [26]:
n_factors = 5
user_factors = torch.randn(n_users,n_factors)
movie_factors = torch.randn(n_movies,n_factors)

Collaborative filtering from scratch

In [35]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        return (users * movies).sum(dim = 1)  #prediction of movie rating
        
        

In [36]:
x, y, = dls.one_batch()
y.shape

torch.Size([64, 1])

Creation of model

In [37]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())


Training the model

In [38]:
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,1.340368,1.337646,00:06
1,1.033712,1.088351,00:05
2,0.889612,0.989082,00:05
3,0.793524,0.901211,00:05
4,0.767524,0.877259,00:05


Making the prediction range between 0 and 5

In [39]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors,y_range = (0,5)):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
        self.y_range = y_range
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        return sigmoid_range((users * movies).sum(dim = 1),*self.y_range)  #prediction of movie rating constructed to be within the range of 0 and 5
        
        

In [42]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.943368,0.987603,00:05
1,0.739267,0.917041,00:06
2,0.524593,0.909534,00:05
3,0.436818,0.907412,00:06
4,0.420192,0.904500,00:05


Including the bias for both user and movie

In [47]:

class DotProduct(Module):
    def __init__(self,n_users,n_movies,n_factors,y_range = (0,5)):
        self.user_factors = Embedding(n_users,n_factors) #creation of embedding of user factor
        self.movie_factors = Embedding(n_movies, n_factors) #creation of embedding of movie factor
        self.user_bias = Embedding(n_users,1)
        self.movie_bias = Embedding(n_movies,1)
        self.y_range = y_range
    def forward(self,x):  #here x represents the data pair of user id and movie title
        users = self.user_factors(x[:,0])  #here the first column represents the userID
        movies = self.movie_factors(x[:,1])  #the second column represents the movie title
        rating = (users * movies).sum(dim = 1, keepdim = True)  #prediction of movie rating 
        rating += self.user_bias(x[:,0]) + self.movie_bias(x[:,1])  #addition of both user bias and movie bias
        return sigmoid_range(rating,*self.y_range)  #using sigmoid function to construct the rating of a movie to be within 0 and 5 
        

In [48]:
model = DotProduct(n_users, n_movies, 50)  #we are using 50 number of factors
learn = Learner(dls, model , loss_func = MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.903987,0.948290,00:06
1,0.654668,0.880255,00:06
2,0.474587,0.903127,00:06
3,0.385534,0.907306,00:06
4,0.348165,0.907174,00:06


As we can observe from the above result that our model is overfitting , cause even though the training error is decreasing, the validation error is increasing. So we must use the regularization to solve this issue.

In [49]:
learn.fit_one_cycle(5,5e-3,wd = 0.1)  #using the weight decay inorder to shrink the value of weight, inorder to prevent overfitting of model

epoch,train_loss,valid_loss,time
0,0.381305,0.908118,00:06
1,0.415450,0.900586,00:06
2,0.382760,0.882960,00:06
3,0.350784,0.873773,00:06
4,0.326903,0.869665,00:06


As we can observe that both the training as well as validation error is decreasing gradually.

Creating our own embedding module

In [53]:
class T(Module):
    def __init__(self):
        self.a = nn.Parameter(torch.ones(3))   #here we are creating a learning parameter for our model
list(T().parameters())       #converting the  pytorch parameters into python list


[Parameter containing:
 tensor([1., 1., 1.], requires_grad=True)]